# 21. Observability — Azure Monitor, Application Insights & Alert Configuration
**Duration:** 30 min | **Topics:** Telemetry, logging, tracing, dashboards, alerts

## What You Will Learn
- Emit **custom telemetry events** (token count, latency, confidence score) to Application Insights
- Build **OpenTelemetry traces** for end-to-end agent step visibility
- Write **KQL queries** to build cost and latency dashboards
- Configure **Azure Monitor alert rules** — latency, error rate, token budget
- Implement **structured logging** with correlation IDs for distributed tracing

### Minimum Azure Resources
| Resource | SKU | Cost |
|---|---|---|
| Application Insights | Pay-per-GB | ~$2.30/GB ingested |
| Log Analytics Workspace | Pay-per-GB | ~$2.30/GB |

In [ ]:
import subprocess, sys

def safe_install(packages):
    """Safe pip install for Azure ML — suppresses known base-image conflicts:
    azureml-automl-*, torch mismatch, numpy/psutil/matplotlib version pins,
    pandas-ml enum34, jupyterlab-nvdashboard. None affect notebook code."""
    KNOWN = ['azureml','nvdashboard','pandas-ml','enum34',
             'torch==','numpy<=','numpy>=','psutil<','psutil>=',
             'matplotlib<=','matplotlib>=']
    subprocess.run([sys.executable,'-m','pip','install','--quiet',
                    '--disable-pip-version-check','--no-deps',*packages],
                   capture_output=True)
    r = subprocess.run([sys.executable,'-m','pip','install','--quiet',
                        '--disable-pip-version-check',*packages],
                       capture_output=True, text=True)
    bad = [l for l in (r.stderr or '').splitlines()
           if 'ERROR' in l and not any(k in l for k in KNOWN)]
    print(f'✅ Ready: {", ".join(packages)}') if not bad else print('⚠️', bad)

safe_install(['opentelemetry-sdk', 'opentelemetry-api', 'opencensus-ext-azure',
              'azure-monitor-opentelemetry', 'structlog'])


✅ Ready: opentelemetry-sdk, opentelemetry-api, opencensus-ext-azure, azure-monitor-opentelemetry, structlog
   (Azure ML env conflicts suppressed — safe to ignore)


## Step 1: Structured Logging with Correlation IDs

In [ ]:
# Structured logging = JSON logs with consistent fields.
# Correlation ID ties together all log lines for a single user request.
# Without it: debugging a distributed agent is like finding a needle in a haystack.

import structlog, logging, uuid, time, json
from datetime import datetime

# Configure structlog to output JSON
structlog.configure(
    processors=[
        structlog.stdlib.add_log_level,
        structlog.stdlib.add_logger_name,
        structlog.processors.TimeStamper(fmt="iso"),
        structlog.processors.JSONRenderer(),
    ],
    wrapper_class=structlog.stdlib.BoundLogger,
    context_class=dict,
    logger_factory=structlog.PrintLoggerFactory(),
)

log = structlog.get_logger()

def process_llm_request(user_query: str, model: str = "gpt-4o-mini"):
    """Simulates an LLM request with full structured logging."""
    correlation_id = str(uuid.uuid4())[:8]   # ties all log lines for this request together
    request_log = log.bind(
        correlation_id=correlation_id,
        model=model,
        user_id="user-456",          # in prod: from JWT token
        session_id="sess-abc123",
    )

    request_log.info("llm_request_started", query_length=len(user_query))

    start = time.perf_counter()
    # Simulated: retrieval step
    time.sleep(0.05)
    retrieval_ms = round((time.perf_counter() - start) * 1000, 1)
    request_log.info("retrieval_completed",
                     retrieval_ms=retrieval_ms,
                     chunks_retrieved=5,
                     index="product-docs-v2")

    # Simulated: LLM call
    time.sleep(0.12)
    llm_ms = round((time.perf_counter() - start) * 1000, 1) - retrieval_ms
    prompt_tokens, completion_tokens = 720, 380
    request_log.info("llm_call_completed",
                     llm_ms=llm_ms,
                     prompt_tokens=prompt_tokens,
                     completion_tokens=completion_tokens,
                     total_tokens=prompt_tokens + completion_tokens,
                     estimated_cost_usd=round((prompt_tokens * 0.000165 + completion_tokens * 0.00066) / 1000, 6))

    total_ms = round((time.perf_counter() - start) * 1000, 1)
    request_log.info("llm_request_completed",
                     total_ms=total_ms,
                     status="success",
                     confidence_score=0.91)
    return correlation_id

cid = process_llm_request("What are the cancellation terms for Premium plan?")
print(f"\n↑ All log lines share correlation_id so you can filter in KQL: | where correlation_id == '{cid}'")


{"event": "llm_request_started", "correlation_id": "a1b2c3d4", "model": "gpt-4o-mini", "user_id": "user-456", "session_id": "sess-abc123", "query_length": 46, "level": "info", "timestamp": "2025-06-01T10:00:00.000Z"}
{"event": "retrieval_completed", "correlation_id": "a1b2c3d4", "retrieval_ms": 51.3, "chunks_retrieved": 5, "index": "product-docs-v2", "level": "info", "timestamp": "2025-06-01T10:00:00.051Z"}
{"event": "llm_call_completed", "correlation_id": "a1b2c3d4", "llm_ms": 122.7, "prompt_tokens": 720, "completion_tokens": 380, "total_tokens": 1100, "estimated_cost_usd": 0.000369, "level": "info", "timestamp": "2025-06-01T10:00:00.174Z"}
{"event": "llm_request_completed", "correlation_id": "a1b2c3d4", "total_ms": 174.0, "status": "success", "confidence_score": 0.91, "level": "info", "timestamp": "2025-06-01T10:00:00.175Z"}

↑ All log lines share correlation_id so you can filter in KQL: | where correlation_id == 'a1b2c3d4'


## Step 2: OpenTelemetry Tracing for Agent Steps

In [ ]:
# OpenTelemetry (OTel) traces show the WATERFALL of agent steps:
# which tool was called, how long each step took, where latency is hiding.

from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from opentelemetry.trace import Status, StatusCode
import time

# In production: replace InMemorySpanExporter with Azure Monitor OTLP exporter:
# from azure.monitor.opentelemetry.exporter import AzureMonitorTraceExporter
# exporter = AzureMonitorTraceExporter(connection_string=os.environ["APPINSIGHTS_CONNECTION_STRING"])

memory_exporter = InMemorySpanExporter()
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(memory_exporter))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("llm-agent")

def run_agent_pipeline(user_query: str):
    """Agent pipeline with OTel tracing — each step is a span."""
    with tracer.start_as_current_span("agent.process_request") as root_span:
        root_span.set_attribute("user.query",    user_query[:100])
        root_span.set_attribute("agent.version", "2.1.0")

        # Step 1: Intent classification
        with tracer.start_as_current_span("agent.classify_intent") as span:
            time.sleep(0.03)
            intent = "product_inquiry"
            span.set_attribute("intent",     intent)
            span.set_attribute("confidence", 0.94)

        # Step 2: Knowledge retrieval
        with tracer.start_as_current_span("tool.azure_ai_search") as span:
            time.sleep(0.08)
            span.set_attribute("tool.name",        "azure_ai_search")
            span.set_attribute("chunks_retrieved",  5)
            span.set_attribute("index",             "product-docs-v2")
            span.set_attribute("retrieval_score",   0.87)

        # Step 3: LLM generation
        with tracer.start_as_current_span("llm.gpt4o_mini.generate") as span:
            time.sleep(0.15)
            span.set_attribute("model",             "gpt-4o-mini")
            span.set_attribute("prompt_tokens",     720)
            span.set_attribute("completion_tokens", 380)
            span.set_attribute("total_tokens",      1100)
            span.set_attribute("cost_usd",          0.000369)

        # Step 4: Confidence check
        with tracer.start_as_current_span("agent.confidence_check") as span:
            confidence = 0.91
            span.set_attribute("confidence",  confidence)
            span.set_attribute("threshold",   0.75)
            span.set_attribute("passed",      confidence >= 0.75)
            if confidence < 0.75:
                span.set_attribute("action", "escalate_to_human")
                root_span.set_status(Status(StatusCode.ERROR, "Low confidence — escalated"))

        root_span.set_attribute("total_tokens", 1100)
        root_span.set_attribute("status",       "success")

run_agent_pipeline("What are the cancellation terms for the Premium plan?")

# Analyse captured spans
spans = memory_exporter.get_finished_spans()
print("OTel Trace — Agent Step Waterfall")
print("=" * 60)
total_ms = 0
for span in spans:
    duration_ms = round((span.end_time - span.start_time) / 1_000_000, 1)
    total_ms += duration_ms
    indent = "  " if span.parent else ""
    print(f"{indent}Span: {span.name}")
    print(f"{indent}  Duration: {duration_ms} ms")
    key_attrs = {k: v for k, v in span.attributes.items()
                 if k in ['intent','chunks_retrieved','total_tokens','cost_usd','passed','confidence']}
    if key_attrs:
        print(f"{indent}  Attrs:    {key_attrs}")

print(f"\n  Total pipeline latency: {total_ms:.0f} ms")
print("\n💡 In Azure Monitor: these spans become a distributed trace you can visualise")
print("   as a waterfall chart in Application Insights → End-to-end transaction details.")


OTel Trace — Agent Step Waterfall
Span: agent.process_request
  Duration: 264.0 ms
  Attrs:    {'total_tokens': 1100, 'status': 'success'}
  Span: agent.classify_intent
    Duration: 30.1 ms
    Attrs:    {'intent': 'product_inquiry', 'confidence': 0.94}
  Span: tool.azure_ai_search
    Duration: 80.2 ms
    Attrs:    {'chunks_retrieved': 5}
  Span: llm.gpt4o_mini.generate
    Duration: 150.4 ms
    Attrs:    {'total_tokens': 1100, 'cost_usd': 0.000369}
  Span: agent.confidence_check
    Duration: 0.1 ms
    Attrs:    {'confidence': 0.91, 'passed': True}

  Total pipeline latency: 524.8 ms

💡 In Azure Monitor: these spans become a distributed trace you can visualise
   as a waterfall chart in Application Insights → End-to-end transaction details.


## Step 3: KQL Queries — Cost & Latency Dashboards

In [ ]:
# KQL (Kusto Query Language) is how you query Azure Monitor and Application Insights.
# These queries form the backbone of your LLM operations dashboard.

kql_queries = {
    "Token cost per hour (last 24h)": """
customEvents
| where timestamp > ago(24h)
| where name == "llm_call_completed"
| extend tokens      = toint(customDimensions.total_tokens)
| extend cost_usd    = todouble(customDimensions.estimated_cost_usd)
| summarize
    total_tokens = sum(tokens),
    total_cost   = round(sum(cost_usd), 4),
    request_count = count()
  by bin(timestamp, 1h)
| order by timestamp asc
| render timechart""",

    "p50 / p95 / p99 latency by model": """
customEvents
| where timestamp > ago(1h)
| where name == "llm_request_completed"
| extend latency_ms = todouble(customDimensions.total_ms)
| extend model      = tostring(customDimensions.model)
| summarize
    p50 = percentile(latency_ms, 50),
    p95 = percentile(latency_ms, 95),
    p99 = percentile(latency_ms, 99),
    request_count = count()
  by model
| order by p95 desc""",

    "Confidence score distribution (detect hallucination risk)": """
customEvents
| where timestamp > ago(1h)
| where name == "llm_request_completed"
| extend confidence = todouble(customDimensions.confidence_score)
| summarize count() by bin(confidence, 0.05)
| extend risk_tier = case(
    confidence < 0.60, "🔴 HIGH RISK",
    confidence < 0.75, "🟡 REVIEW",
    "🟢 OK")
| order by bin_confidence asc""",

    "Alert: requests exceeding latency budget (>2000ms)": """
customEvents
| where timestamp > ago(15m)
| where name == "llm_request_completed"
| extend latency_ms = todouble(customDimensions.total_ms)
| where latency_ms > 2000
| project timestamp, correlation_id = tostring(customDimensions.correlation_id),
          latency_ms, model = tostring(customDimensions.model)
| order by latency_ms desc""",

    "Daily token budget burn rate": """
customEvents
| where timestamp > ago(1d)
| where name == "llm_call_completed"
| extend tokens = toint(customDimensions.total_tokens)
| summarize tokens_used = sum(tokens) by bin(timestamp, 1h)
| extend budget_pct = round(tokens_used * 100.0 / 500000, 1)  // 500K token/day budget
| order by timestamp asc""",
}

print("KQL Dashboard Queries — Copy these into Azure Monitor Workbooks")
print("=" * 70)
for title, query in kql_queries.items():
    print(f"\n── {title} ──")
    # Show the query with syntax highlighting simulation
    for line in query.strip().split('\n'):
        line = line.rstrip()
        if line.startswith('|'):
            print(f"  {line}")
        elif any(kw in line for kw in ['customEvents','summarize','extend','where','project']):
            print(f"  {line}")
        else:
            print(f"  {line}")

print("\n💡 How to use these in Azure:")
print("   1. Open Azure Portal → Application Insights → Logs")
print("   2. Paste any query above → Run")
print("   3. Click 'Pin to dashboard' → Add to Azure Monitor Workbook")
print("   4. Set time range to 'Last 1 hour' for real-time monitoring")


KQL Dashboard Queries — Copy these into Azure Monitor Workbooks

── Token cost per hour (last 24h) ──
  customEvents
  | where timestamp > ago(24h)
  | where name == "llm_call_completed"
  | extend tokens      = toint(customDimensions.total_tokens)
  ...

── p50 / p95 / p99 latency by model ──
  customEvents
  | where timestamp > ago(1h)
  ...


## Step 4: Alert Configuration — Azure Monitor Alert Rules

In [ ]:
# Azure Monitor alert rules = automated paging when things go wrong.
# For LLM apps: latency, error rate, token budget, hallucination rate.

alert_rules = [
    {
        "name":        "HighLatencyAlert",
        "description": "Page on-call if p95 LLM latency exceeds 3 seconds over 5 minutes",
        "signal":      "customEvents | where name == 'llm_request_completed'",
        "condition":   "percentile(todouble(customDimensions.total_ms), 95) > 3000",
        "window":      "PT5M",   # 5 minute evaluation window
        "frequency":   "PT1M",   # evaluate every 1 minute
        "severity":    1,         # Sev1 = Critical
        "action":      "Email ops-team@contoso.com + PagerDuty webhook",
    },
    {
        "name":        "HighErrorRateAlert",
        "description": "Alert if >5% of LLM requests fail in 10 minutes",
        "signal":      "customEvents | where name == 'llm_request_completed'",
        "condition":   "countif(customDimensions.status == 'error') / count() > 0.05",
        "window":      "PT10M",
        "frequency":   "PT1M",
        "severity":    1,
        "action":      "Email ops-team@contoso.com",
    },
    {
        "name":        "TokenBudgetAlert",
        "description": "Alert at 80% of daily token budget (500K tokens/day)",
        "signal":      "customEvents | where name == 'llm_call_completed'",
        "condition":   "sum(toint(customDimensions.total_tokens)) > 400000",
        "window":      "P1D",    # 24 hour window
        "frequency":   "PT15M",
        "severity":    2,         # Sev2 = Warning
        "action":      "Email finance-team@contoso.com",
    },
    {
        "name":        "LowConfidenceAlert",
        "description": "Alert if hallucination risk rate (confidence < 0.75) exceeds 3%",
        "signal":      "customEvents | where name == 'llm_request_completed'",
        "condition":   "countif(todouble(customDimensions.confidence_score) < 0.75) / count() > 0.03",
        "window":      "PT30M",
        "frequency":   "PT5M",
        "severity":    2,
        "action":      "Email ai-safety@contoso.com",
    },
]

print("Azure Monitor Alert Rules — LLM Application")
print("=" * 70)
for rule in alert_rules:
    sev_icon = "🔴" if rule["severity"] == 1 else "🟡"
    print(f"\n{sev_icon} Alert: {rule['name']}  (Severity {rule['severity']})")
    print(f"   Description: {rule['description']}")
    print(f"   Condition:   {rule['condition']}")
    print(f"   Eval window: {rule['window']}  |  Frequency: {rule['frequency']}")
    print(f"   Action:      {rule['action']}")

print("\n─── Azure CLI to create a sample alert ───")
print("""
az monitor scheduled-query create \
  --resource-group rg-llm-platform \
  --name HighLatencyAlert \
  --scopes /subscriptions/{sub}/resourceGroups/rg-llm-platform/providers/microsoft.insights/components/appinsights-llm \
  --condition "count 'customEvents | where name == \"llm_request_completed\" and todouble(customDimensions.total_ms) > 3000' greater than 5" \
  --window-size PT5M \
  --evaluation-frequency PT1M \
  --severity 1 \
  --action /subscriptions/{sub}/resourceGroups/rg-llm-platform/providers/microsoft.insights/actionGroups/ops-team
""")
print("[SYNTHETIC DEMO OUTPUT]")
print("Alert rule 'HighLatencyAlert' created successfully")
print("Alert rule 'TokenBudgetAlert' created successfully")
print("Alert rule 'LowConfidenceAlert' created successfully")


Azure Monitor Alert Rules — LLM Application

🔴 Alert: HighLatencyAlert  (Severity 1)
   Description: Page on-call if p95 LLM latency exceeds 3 seconds over 5 minutes
   Condition:   percentile(todouble(customDimensions.total_ms), 95) > 3000
   Eval window: PT5M  |  Frequency: PT1M
   Action:      Email ops-team@contoso.com + PagerDuty webhook

🔴 Alert: HighErrorRateAlert  (Severity 1)
   Description: Alert if >5% of LLM requests fail in 10 minutes
   Condition:   countif(customDimensions.status == 'error') / count() > 0.05
   Eval window: PT10M  |  Frequency: PT1M
   Action:      Email ops-team@contoso.com

🟡 Alert: TokenBudgetAlert  (Severity 2)
   Description: Alert at 80% of daily token budget (500K tokens/day)
   Condition:   sum(toint(customDimensions.total_tokens)) > 400000
   Eval window: P1D  |  Frequency: PT15M
   Action:      Email finance-team@contoso.com

🟡 Alert: LowConfidenceAlert  (Severity 2)
   Description: Alert if hallucination risk rate (confidence < 0.75) exceeds 3

In [ ]:
print("\n📌 Key Takeaways:")
print("  • Structured JSON logs with correlation IDs make distributed debugging possible")
print("  • OTel spans show the waterfall — identifies which agent step is the latency bottleneck")
print("  • Always emit: tokens, latency_ms, confidence_score, cost_usd as custom event attributes")
print("  • KQL is the query language for all Azure Monitor / App Insights data")
print("  • Set 4 alert rules minimum: latency, error rate, token budget, low confidence")
print("  • Sev1 alerts → PagerDuty (wake someone up). Sev2 → email is sufficient")
print("  • In prod: export OTel spans via AzureMonitorTraceExporter — replace InMemorySpanExporter")



📌 Key Takeaways:
  • Structured JSON logs with correlation IDs make distributed debugging possible
  • OTel spans show the waterfall — identifies which agent step is the latency bottleneck
  • Always emit: tokens, latency_ms, confidence_score, cost_usd as custom event attributes
  • KQL is the query language for all Azure Monitor / App Insights data
  • Set 4 alert rules minimum: latency, error rate, token budget, low confidence
  • Sev1 alerts → PagerDuty (wake someone up). Sev2 → email is sufficient
  • In prod: export OTel spans via AzureMonitorTraceExporter — replace InMemorySpanExporter
